In [1]:
from pathlib import Path
import os
import shutil


In [2]:
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

ipython = get_ipython()


@register_cell_magic
def pybash(line, cell):
    cell_replaced = eval("f" + repr(cell))
    # print("Evaluating:\n{}\n-----------".format(cell_replaced))
    ipython.run_cell_magic('bash', '', cell_replaced)

In [3]:
project_dir = Path(os.getcwd()).parent.parent
install_dir = project_dir / "install"
log_dir = project_dir / "logs" / "dlio"
data_dir = Path("/p/lustre5/haridev/dlio_demo")
output_dir = project_dir / "output" / "dlio"
print("Directories created:")
for name, path in [("Install Directory", install_dir), 
                   ("Log Directory", log_dir), 
                   ("Data Directory", data_dir), 
                   ("Output Directory", output_dir)]:
    print(f"{name}: {path}")

Directories created:
Install Directory: /usr/WS2/haridev/dftracer-demo/install
Log Directory: /usr/WS2/haridev/dftracer-demo/logs/dlio
Data Directory: /p/lustre5/haridev/dlio_demo
Output Directory: /usr/WS2/haridev/dftracer-demo/output/dlio


In [4]:

for dir_path in [log_dir, data_dir, output_dir]:
    if dir_path.exists():
        for item in dir_path.iterdir():
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
    dir_path.mkdir(parents=True, exist_ok=True)
print("Cleaned and created fresh folders for log, data, and output.")

Cleaned and created fresh folders for log, data, and output.


In [5]:
import os

import importlib.util

spec = importlib.util.find_spec("dftracer")
if spec and spec.origin:
    print("dftracer module path:", spec.origin)
    dftracer_folder = os.path.dirname(spec.origin)
    print("dftracer folder:", dftracer_folder)
else:
    print("dftracer module not found.")

dftracer module path: /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dftracer/__init__.py
dftracer folder: /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dftracer


[Documentation](https://dftracer.readthedocs.io/en/latest/api.html)

In [ ]:
%%pybash
# DFTracer environment variables:
echo "Configuring DFTracer"

# DFTRACER_INC_METADATA: Include or exclude metadata (default 0)
export DFTRACER_INC_METADATA=1

# DFTRACER_ENABLE: Enable or Disable DFTracer (default 0).
export DFTRACER_ENABLE=1

export DFTRACER_TRACE_COMPRESSION=0

echo "Activating environment"
source {project_dir}/demo/dlio/setup_env.sh {install_dir} 2> /dev/null


echo "Running DLIO with DFTracer"
flux run -n 2 -o fastload -q pbatch {install_dir}/bin/dlio_benchmark workload=unet3d_a100 ++workload.workflow.generate_data=True hydra.run.dir={output_dir}/unet3d_a100/ ++workload.output.folder={output_dir}/unet3d_a100/ ++workload.output.folder={output_dir}/unet3d_a100/ ++workload.dataset.num_files_train=32 ++workload.dataset.record_length_bytes=1048576 ++workload.dataset.data_folder={data_dir}/unet3d_a100/data ++workload.checkpoint.checkpoint_folder={data_dir}/unet3d_a100/checkpoint ++workload.train.epochs=1 > {output_dir}/log.txt 2> {output_dir}/error.txt
echo "Finished running DLIO with DFTracer"

Configuring DFTracer
Activating environment
Running DLIO with DFTracer


In [ ]:
%%pybash
{install_dir}/bin/dftracer_pgzip -d {output_dir}/unet3d_a100

10/08/2025 01:44:37  Found 18 .pfw files to process.
10/08/2025 01:44:37  Processed 1/18 files.
10/08/2025 01:44:37  Gzip Completed. Processed 0/18 files.


In [8]:
import glob

pfw_files = glob.glob(str(output_dir/ "unet3d_a100" / "*.pfw.gz"))
if pfw_files:
    print("Found .pfw.gz files:")
    for f in pfw_files:
        print(f)
else:
    print("No .pfw.gz files found in", log_dir)


Found .pfw.gz files:
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-0262698dba4e76ef-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-049cbd766c3cec5b-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-0e4a7c09f5131231-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-0-of-2.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-1c29139c39441d13-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-1e495efbab10b623-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-1-of-2.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-232be2f5ac9a4eb1-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-344bda897d9dc0b6-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-46521cb5dc0d6706-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-4968acdb3969208c-app.pfw.gz
/usr/WS2/haridev/dftracer-de

In [ ]:
%%pybash
{install_dir}/bin/dftracer_split -n unet3d -f -d {output_dir}/unet3d_a100 -o {output_dir}/unet3d_a100/compact

Arguments:
  App name: unet3d
  Override: 1
  Data dir: /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100
  Output dir: /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/compact
  Chunk size: 1024
10/08/2025 01:49:05 Found zindex executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zindex
10/08/2025 01:49:05 Found zq executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zq
10/08/2025 01:49:05 sqlite3 exists
10/08/2025 01:49:05  Number of *.pfw* files in /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100: 36
10/08/2025 01:49:05  Found zindex executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zindex
10/08/2025 01:49:05  Removing existing indices as override is passed.
10/08/2025 01:49:05  Removed index file: /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-0262698dba4e76ef-app.pfw.gz.zindex
10/08/2025 01:49:05  Remov

rm: cannot remove '/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/counting.bak': No such file or directory


10/08/2025 01:49:05 Completed collecting size .00097560 17 of 18                               
10/08/2025 01:49:05 Finished collecting data from 18 tasks
10/08/2025 01:49:06 Scheduled chunks: 0
10/08/2025 01:49:06 Total chunks: 1
10/08/2025 01:49:06 Start processing chunks
10/08/2025 01:49:06 Chunk 1 out of 1 done with size 8.26655471 MB, path = /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/compact/unet3d-1.pfw                               
10/08/2025 01:49:06 All chunks processed
10/08/2025 01:49:06 re-index split files
10/08/2025 01:49:06  Number of *.pfw* files in /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/compact: 1
10/08/2025 01:49:06  Found zindex executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zindex
10/08/2025 01:49:06  Removing existing indices as override is passed.
10/08/2025 01:49:06  Compressing file unet3d-1
10/08/2025 01:49:06  Created index for compressed file /usr/WS2/haridev/dftracer-demo/out

10/08/2025 01:49:08 Original lines count 36193 matches split lines count 36193
10/08/2025 01:49:08 Done re-index of split files


In [13]:
!gzip -dc {output_dir}/unet3d_a100/compact/*.pfw.gz | (head -n 10; echo "..."; tail -n 5)

[
{"id":938,"name":"thread_name","cat":"dftracer","pid":515263,"tid":515263,"ph":"M","args":{"hhash":"a2dc02804a75442c","name":"515263","value":"thread_name"}}
{"id":939,"name":"start","cat":"dftracer","pid":515263,"tid":515263,"ts":1754815202228229,"dur":0,"ph":"X","args":{"hhash":"a2dc02804a75442c","p_idx":937,"level":8,"ppid":514963,"date":"Sun Aug 10 01:40:02 2025","cmd_hash":"097abd914d62cdfb","exec_hash":"de04e44a7b1b2401","version":"v1.0.14-10-g6d9243c"}}
{"id":937,"name":"fork","cat":"POSIX","pid":515263,"tid":515263,"ts":1754815202203159,"dur":25100,"ph":"X","args":{"hhash":"a2dc02804a75442c","p_idx":927,"level":7,"ret":0}}
{"id":943,"name":"NPZReader.__init__","cat":"reader","pid":515263,"tid":515263,"ts":1754815202657116,"dur":53,"ph":"X","args":{"hhash":"a2dc02804a75442c","p_idx":942,"level":10,"epoch":"3"}}
{"id":942,"name":"TorchDataset.worker_init","cat":"data_loader","pid":515263,"tid":515263,"ts":1754815202249541,"dur":407679,"ph":"X","args":{"hhash":"a2dc02804a75442c"

In [4]:
from dfanalyzer import init_with_hydra
dfa = init_with_hydra(
    hydra_overrides=[
        f"trace_path={output_dir}/unet3d_a100/compact/",
        f"analyzer/preset=dlio",
        
    ]
)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36327 instead
  warnings.warn(


In [5]:
dfa.client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:36327/status,
Dashboard: http://127.0.0.1:36327/status,Workers: 12
Total threads: 96,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36299,Workers: 12
Dashboard: http://127.0.0.1:36327/status,Total threads: 96
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46497,Total threads: 8
Dashboard: http://127.0.0.1:35643/status,Memory: 0 B
Nanny: tcp://127.0.0.1:44813,


In [6]:
res = dfa.analyze_trace()
dfa.output.handle_result(res)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dask_expr/_collection.py:5063: FutureWarning: from_legacy_dataframe is deprecated and will be removed in a future release. The legacy implementation as a whole is deprecated and will be removed, making this method unnecessary.
  warnings.warn(
/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dfanalyzer/output.py:112: RuntimeWarning: invalid value encountered in scalar divide
  ops=float('nan') if pd.isna(time) else float(count / time),
/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dfanalyzer/output.py:113: RuntimeWarning: invalid value encountered in scalar divide
  bandwidth=float('nan') if pd.isna(time) or pd.isna(size) else float(size / time),


                                                Time Period Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric                                                                    ┃ Unit            ┃             Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Job Time                                                                  │ seconds         │            29.880 │
│ Total Count                                                               │ count           │            35,875 │
│ Total Files                                                               │ count           │                 0 │
│ Total Nodes                                                               │ count           │                 0 │
│ Total Processes                                                           │ count           │                18 │
│ App Count                                                                 │ count           │                 2 │
│ Training Count                                                            │ count           │                 2 │
│ Compute Count                                                             │ count           │                 4 │
│ Fetch Data Count                                                          │ count           │                34 │
│ Data Loader Count                                                         │ count           │                36 │
│ Data Loader Fork Count                                                    │ count           │                32 │
│ Reader Count                                                              │ count           │               112 │
│ Reader POSIX (Lustre) Count                                               │ count           │            35,617 │
│ Reader POSIX (Lustre) Size                                                │ MB              │         15821.551 │
│ Reader POSIX (Lustre) Bandwidth                                           │ MB/s            │          1724.056 │
│ Reader POSIX (Lustre) Avg Transfer Size                                   │ MB              │             0.444 │
│ Checkpoint Count                                                          │ count           │                 1 │
│ Checkpoint POSIX (Lustre) Count                                           │ count           │                 3 │
│ Other POSIX Count                                                         │ count           │                32 │
└───────────────────────────────────────────────────────────────────────────┴─────────────────┴───────────────────┘
                                          Layer Breakdown (w/ overlap %)                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Layer                       ┃       Time (s) ┃            Ops ┃   Ops/sec ┃        Size (MB) ┃ Bandwidth (MB/s) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ App                         │   9.449 (----) │       2 (----) │     0.212 │                - │                - │
│ Training                    │   9.403 (----) │       2 (----) │     0.213 │                - │                - │
│ Compute                     │   2.544 (----) │       4 (----) │     1.572 │                - │                - │
│ Fetch Data                  │   6.734 (  0%) │      34 (  0%) │     5.049 │                - │                - │
│ Data Loader                 │  17.672 (  5%) │      36 (  8%) │     2.037 │                - │                - │
│ Data Loader Fork            │   0.105 (  0%) │      32 (  0%) │   304.539 │                - │                - │
│ Reader                      │  16.772 (  6%) │     112